# DHIS2 Map Data Explorer

This workflow receives the selected map data and boundary geometry from DHIS2 Analysis. It creates a GeoPackage boundary file, keeps the GeoJSON and tabular CSV, and displays numeric values as a map layer.

In [ ]:
data_dir <- ""
out_dir <- ""

In [ ]:
library(sf)
library(jsonlite)

input_geojson <- file.path(data_dir, "dhis2-map.geojson")
input_csv <- file.path(data_dir, "dhis2-map.csv")
metadata <- fromJSON(file.path(data_dir, "metadata.json"), simplifyVector = FALSE)
map_data <- st_read(input_geojson, quiet = TRUE)

if (!"value" %in% names(map_data)) {
  stop("The DHIS2 export has no value field.")
}
map_data$value <- suppressWarnings(as.numeric(map_data$value))
dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

file.copy(input_geojson, file.path(out_dir, "dhis2-map.geojson"), overwrite = TRUE)
file.copy(input_csv, file.path(out_dir, "dhis2-map.csv"), overwrite = TRUE)
file.copy(file.path(data_dir, "metadata.json"), file.path(out_dir, "metadata.json"), overwrite = TRUE)
st_write(map_data, file.path(out_dir, "dhis2-map.gpkg"), layer = "dhis2_map", delete_dsn = TRUE, quiet = TRUE)

item_name <- metadata$dataItem
if (is.null(item_name) || !nzchar(item_name)) item_name <- metadata$name
if (is.null(item_name) || !nzchar(item_name)) item_name <- "DHIS2 value"
period_label <- metadata$periodLabel
if (is.null(period_label) || !nzchar(period_label)) period_label <- "Selected period"
layers <- list()
if (any(is.finite(map_data$value))) {
  layers <- list(list(
    id = "dhis2-map",
    title = item_name,
    kind = "choropleth",
    file = "dhis2-map.geojson",
    fields = list(list(field = "value", label = period_label, type = "numeric"))
  ))
}
write_json(list(layers = layers), file.path(out_dir, "layers.json"), auto_unbox = TRUE, pretty = TRUE, na = "null")